# Lesson 8 | How do we know the hardware is correct?

Plausible-looking RTL is not verified. Today asks:
> **Without a physical FPGA board, how can we drive RTL, observe outputs, and make errors fail automatically?**

Primary concept: **hardware simulation and the testbench**.


## 1. Terms

**simulation:** software executes HDL logic/timing semantics; it is not a physical chip.

**testbench:** verification HDL that generates clock/reset/input and checks outputs.

**waveform:** signal values plotted against simulation time.

The checked module is often the **Device Under Test (DUT)**.


## 2. A testbench does not become final hardware

A testbench may use `#5`, logging, `$fatal`, and terminate simulation. Those are verification behaviors. Design RTL is the part intended for synthesis into hardware.


## 3. Write the oracle before looking at a waveform

A waveform should not be judged by “looks about right.” Compute expected values from the contract first:


In [ ]:
state=0; threshold=4
print('cycle | input | before | candidate | spike | after')
for cycle,current in enumerate([1,1,1,1,2,2]):
    before=state; candidate=before+current; spike=candidate>=threshold
    state=0 if spike else candidate
    print(f'{cycle:5d} | {current:5d} | {before:6d} | {candidate:9d} | {int(spike):5d} | {state:5d}')


## 4. Self-checking testbench

`tb/learning/tutorial_if_neuron_tb.sv` compares `membrane_v` / `spike` after every edge. Mismatch calls `$fatal`; full success prints `PASS lesson08 tutorial_if_neuron`.


## 5. Run

When Icarus Verilog is available, the next cell actually compiles and simulates. Otherwise it explicitly reports that simulation did not run.


In [ ]:
from pathlib import Path
import shutil, subprocess, tempfile

def repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'pyproject.toml').exists() and (p/'lessons').exists(): return p
    raise FileNotFoundError('Run inside FPGA-FlyBrain')

root=repo_root(); iverilog=shutil.which('iverilog'); vvp=shutil.which('vvp')
if not (iverilog and vvp):
    print('Icarus Verilog not found; RTL simulation did not run.')
else:
    with tempfile.TemporaryDirectory() as td:
        td=Path(td); out=td/'l8.out'
        subprocess.run([iverilog,'-g2012','-o',str(out),str(root/'rtl/learning/tutorial_if_neuron.sv'),str(root/'tb/learning/tutorial_if_neuron_tb.sv')],check=True,cwd=td)
        r=subprocess.run([vvp,str(out)],check=True,text=True,capture_output=True,cwd=td)
        print(r.stdout.strip())
        print('A temporary tutorial_if_neuron.vcd waveform was generated during the run.')


## 6. Waveform / VCD

The testbench uses `$dumpfile/$dumpvars`; a common format is **Value Change Dump (VCD)**. Use a waveform to answer when input was stable, which cycle owns the spike, and from which edge reset state becomes visible.


## 7. Which layer do you inspect first on failure?

1. Is the contract/oracle correct?
2. Does the testbench drive/sample at the intended time?
3. Is the RTL combinational or register update wrong?

Do not randomly edit RTL until a broken test turns green.


## 8. Try It / AI Task / Human Check

On a temporary copy, change `>=` to `>`. Predict the boundary failure, run it, then restore the code.

Ask AI for evidence-based inspection points from a failure message, not a blind patch.

Without AI, explain simulation vs FPGA, testbench vs design, and the complementary roles of self-checking tests and waveforms.


## 9. Engineering Handoff / Project Trace

This establishes RMD-005/RMD-005A verification habits. Formal LIF RTL will use Python fixed-point vectors as its direct oracle; this tutorial does not replace that step.

- Lesson: `LSN-008`
- Mapping: `RMD-005 / RMD-005A`
- Prepared level: `L3 RTL unit simulation`


## 10. Exit Ticket

You can read small RTL, distinguish combinational logic from registers, explain a self-checking testbench and waveform, and avoid confusing “simulation passed” with “runs on FPGA.”
